In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import re
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch.utils.data import Dataset, DataLoader
from transformers.utils import logging
from lightgbm import LGBMRegressor
import lightgbm as lgb


In [ ]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = 'microsoft/deberta-v3-small'
    tokenizer = None
    max_length = 512
    batch_size = 16
    epochs = 10
    learning_rate = 5e-5
    lgbm_lr = 0.03
    weight_decay = 0.01
    task = 'regression'

In [ ]:
#Quadratic Weighted Kappa score
def get_score(y_true, y_pred):
    return cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

In [ ]:
#Sets random to 42
def seed_everything(seed: int=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    
seed_everything(42)

In [ ]:
path = kagglehub.competition_download(cfg.competition)

In [ ]:
train_df = pd.read_csv(f'{path}/train.csv')

In [ ]:
train_df['word_count'] = train_df['full_text'].str.split(' ').str.len()

train_df['num_sentences'] = train_df['full_text'].apply(lambda x: len([s for s in re.split(r'[.!?]+', str(x)) if s.strip()]))

train_df['sentence_length'] = train_df[['num_sentences', 'word_count']].apply(lambda row: row['word_count']/row['num_sentences'], axis=1)

train_df['num_para'] = train_df['full_text'].apply(lambda x: len([p for p in str(x).split('\n') if p.strip()]))

train_df['para_length'] = train_df [['num_para', 'word_count']].apply(lambda row: row['word_count']/row['num_para'], axis=1)

train_df['long_essay'] = (train_df['word_count']>2500).astype(int)

In [ ]:
#Label y starting from 0 so classifier works
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1

In [ ]:
#Cross Validation
train_df['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['score'])):
    train_df.loc[valid_idx, 'fold'] = fold

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)

In [ ]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=cfg.max_length):
        self.X = data['full_text']
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.float)
        return item


In [ ]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [ ]:
class EssayModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        #MLP NN
        self.classifier = nn.Sequential(
            nn.Linear(self.model.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)

        )

    def forward(self, input_ids, attention_mask, features):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        mp_output = self.mp(output.last_hidden_state, attention_mask)

        #Add feature engineered outputs
        combined_output = torch.cat([mp_output, features], dim=1)
        logits = self.classifier(combined_output)
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#Get out of fold predictions on validation sets based on DeBERTa small model
oof_pred = np.zeros(len(train_df))
for fold in range(5):
    valid_idx = train_df[train_df['fold']==0].index
    valid_df = train_df.loc[valid_idx].reset_index(drop=True)

    valid_dataset = EssayDataset(valid_df, tokenizer)
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False)

    logging.set_verbosity_error()
    deberta_model = EssayModel().to(device)
    deberta_model.load_state_dict(
        torch.load(f'best_deberta_model_fold_0.pth', map_location=device)
    )
    deberta_model.eval()

    preds = []
    with torch.no_grad():
            for batch in valid_loader:
                logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device)).squeeze(-1)
                preds.extend(logits.cpu().numpy())
                
    oof_pred[valid_idx] = preds
    del deberta_model

train_df['mlp_oof'] = oof_pred

In [ ]:

features = ['mlp_oof', 'word_count', 'num_sentences', 'sentence_length', 'num_para', 'para_length', 'long_essay']
X = train_df[features]
y = train_df['score']

lgb_model = LGBMRegressor(
    objective=cfg.task,
    learning_rate=cfg.lgbm_lr,
    n_estimators=300,
    num_leaves=16,s
    max_depth=4,
    random_state=42
)

lgb_model.fit(X, y)

In [ ]:
base_qwk = get_score(y, np.round(train_df['mlp_oof']).clip(0,5).astype(int))

lgb_preds = lgb_model.predict(X)
lgb_qwk = get_score(y, np.round(lgb_preds).clip(0,5).astype(int))
print('Base QWK:', base_qwk)
print('LGB Stack QWK:', lgb_qwk)